##Paso 0. Cargar las librerías y paquetes a utilizar

In [ ]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

##Lectura de datos

In [ ]:
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive/" + "DM/datos/propiedades2026"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col='id')
df_ent.head(3)

## 1. Entender los datos (AID)

## 2. Limpiar y transformar los datos (DM)

In [ ]:
df_ent["property_type"] = df_ent["property_type"].replace('departamentos','departamento')
df_ent["property_type"] = df_ent["property_type"].replace('casas','casa')

## 2.1. Filtrado de datos

In [ ]:
# selección de datos
tiene_precio = df_ent["price"].notna()
moneda_dolares = df_ent["currency_type"] == 'dolares'
localidades = df_ent["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA'])
tipo_operacion = df_ent["operation_type"] == 'venta'
tipo_propiedad = df_ent["property_type"].isin(["departamento", "casa", "departamentos", "casas"])

df_ent = df_ent.loc[tiene_precio & moneda_dolares & localidades & tipo_operacion & tipo_propiedad]
df_ent.shape

##2.4. Creación de nuevos atributos

In [ ]:
# creamos otras variables
texto = (
    df_ent["features"].fillna("").astype(str) + " " +
    df_ent["description"].fillna("").astype(str)
)

# función para extraer números
def extraer(patron):
    return pd.to_numeric(texto.str.extract(patron, expand=False), errors="coerce")

df_ent["bedrooms"] = extraer(r"(\d+)\s*(?:dormitorios?|habitaciones?)")
df_ent["bathrooms"] = extraer(r"(\d+)\s*(?:baños?|banos?)")
df_ent["rooms"] = extraer(r"(\d+)\s*(?:ambientes?|rooms?)")

df_ent["surface_total_m2"] = extraer(r"(\d+(?:[.,]\d+)?)\s*m²?")
df_ent["surface_covered_m2"] = extraer(r"(\d+(?:[.,]\d+)?)\s*m²?\s*(?:cubiertos?|cubierta)")


In [ ]:
# esta celda es equivalente (maso) a la anterior
# 1 dormitorio
# 2 dormitorios
# 1 habitación
# 2 habitaciones

df_ent["bedrooms"] = 0

for i in range(1, 10+1):
    df_ent[df_ent["features"].str.contains(f"{i} dormi"), "bedrooms"] = i
    df_ent[df_ent["features"].str.contains(f"{i} habitaci"), "bedrooms"] = i

In [ ]:
# arreglar las varibles creadas
df_ent.loc[:, "bedrooms"] = df_ent["bedrooms"].fillna(df_ent["rooms"] - 1)
df_ent.loc[:, "rooms"] = df_ent["rooms"].fillna(df_ent["bedrooms"] + 1)
df_ent.loc[:, "bathrooms"] = df_ent["bathrooms"].fillna(df_ent["rooms"] - df_ent["bedrooms"])

df_ent["surface_total_m2"] = df_ent["surface_total_m2"].fillna(df_ent["surface_covered_m2"])
df_ent["surface_covered_m2"] = df_ent["surface_covered_m2"].fillna(df_ent["surface_total_m2"])

# crear variable de relacion superficie cubierta/superficie total
df_ent["pct_surface_covered"] = df_ent["surface_covered_m2"] / df_ent["surface_total_m2"]

In [ ]:
df_ent.shape, df_ent.columns
df_ent["property_type"].value_counts()

In [ ]:
df_ent["price_m2"] = df_ent["price"] / (df_ent["surface_total_m2"] + 0.001)

gb = df_ent.groupby(by=["property_type", "location_2"])["price_m2"].mean().reset_index()
gb

In [ ]:
df_ent = df_ent[df_ent.columns.drop("price_m2")].merge(gb, on=["property_type", "location_2"], how="left")
df_ent.head(2)

In [ ]:
import seaborn as sns
sns.boxplot(data=df_ent, x="property_type", y="price_m2")
import seaborn as sns

##2.2. Tratamiento de valores atípicos



In [ ]:
# filtro lo que tiene un precio por metro cuadrado sin sentido para calular la estadistica
df_ent = df_ent[(df_ent["price_m2"] >= 300) & (df_ent["price_m2"] <= 9000)]
df_ent.shape

##2.3. Imputación de valores perdidos

In [ ]:
# imputación de valores perdidos
df_ent = df_ent.fillna(0)

##3. Entrenamiento del modelos (AA)- ⛔⛔⛔ NO TOCAR NADA DE ESTA SECCIÓN ⛔⛔⛔

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [ ]:

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Definimos el valor de los hiperparámetros a usar por el modelo
n_estimators = 500
max_depth = 50

# Creamos el modelo a entrenar
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo
_ = reg.fit(X_train, y_train)

# Cálculo del error en entrenamiento (train)
y_pred = reg.predict(X_train)
score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

# Cálculo del error en prueba (test)
y_pred = reg.predict(X_test)
score_test  = sk.metrics.root_mean_squared_error(y_test,  y_pred)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

## 3.1. Análisis de la importancia de variables en el modelo (opcional)

Una manera visual de entender a que variable el modelo le está prestando mayor atención.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feat_importances = pd.Series(reg.feature_importances_, index=X.columns)

# gráfico de barras horizontales
feat_importances.nlargest(10).plot(kind='barh');

## 4.1 Solución para subir Kaggle

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
df_ap.head(2)

In [ ]:
##Incorporo columna calculada en df_ent
df_ap = df_ap.reset_index().merge(gb, on=["property_type", "location_2"], how="left").set_index("id")

df_ap.head()

In [ ]:
# someter el dataset a predecir a las mismas transformaciones de entrenamiento
# creo otras variables
texto = (
    df_ap["features"].fillna("").astype(str) + " " +
    df_ap["description"].fillna("").astype(str)
)

# función para extraer números
def extraer(patron):
    return pd.to_numeric(texto.str.extract(patron, expand=False), errors="coerce")

df_ap["bedrooms"] = extraer(r"(\d+)\s*(?:dormitorios?|habitaciones?)")
df_ap["bathrooms"] = extraer(r"(\d+)\s*(?:baños?|banos?)")
df_ap["rooms"] = extraer(r"(\d+)\s*(?:ambientes?|rooms?)")

df_ap["surface_total_m2"] = extraer(r"(\d+(?:[.,]\d+)?)\s*m²?")
df_ap["surface_covered_m2"] = extraer(r"(\d+(?:[.,]\d+)?)\s*m²?\s*(?:cubiertos?|cubierta)")

#arreglar las varibles creadas
df_ap.loc[df_ap["property_type"] == "Cochera", ["bathrooms", "rooms", "bedrooms"]] = 0
df_ap.loc[:, "bedrooms"] = df_ap["bedrooms"].fillna(df_ap["rooms"] - 1)
df_ap.loc[:, "rooms"] = df_ap["rooms"].fillna(df_ap["bedrooms"] + 1)
df_ap.loc[:, "bathrooms"] = df_ap["bathrooms"].fillna(df_ap["rooms"] - df_ap["bedrooms"])

df_ap["surface_total_m2"] = df_ap["surface_total_m2"].fillna(df_ap["surface_covered_m2"])
df_ap["surface_covered_m2"] = df_ap["surface_covered_m2"].fillna(df_ap["surface_total_m2"])

# crear variable de relacion superficie cubierta/superficie total
df_ap["pct_surface_covered"] = df_ap["surface_covered_m2"] / df_ap["surface_total_m2"]


df_ap.head(5)

In [ ]:
#reentrenamiento del modelo
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']


n_estimators = 500
max_depth = 50

reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo con todos los datos de entrenamiento.csv
reg.fit(X, y)

## 4.2. Generación del archivo para Kaggle

In [ ]:
# Hacemos en df_ap la misma limpieza que en df_ent
#final
df_ap = df_ap.fillna(0)

df_ap = df_ap.select_dtypes('number')

X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
df_ap["price"].to_csv(f"{DIR}/solucion-c3p0.csv")
